## Settings & imports

In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from matplotlib.pyplot import cm
import string
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score, silhouette_samples
import csv
import pickle
import re

In [2]:
dataset = 'Konstytucja_prediction'
book_name = 'app'

In [3]:
import json

with open('../config.json', 'r') as f:
    config = json.load(f)

which_dataset = config["which_dataset"]
preprocessing_method = config["preprocessing_method"] # normalization / logarithm
remove_outer_bool = config["remove_outer"]
how_many_outer_to_remove = config["how_many_outer_to_remove"]
elements_to_keep = config["elements_to_keep"][which_dataset]
elements_to_keep_xrf = config["elements_to_keep_xrf"]

data_path = config["data_path"][which_dataset]
target_path = config["target_path"][which_dataset]
figures_path = config["figures_path"]
models_path = config["models_path"][which_dataset]
classes_path = config["classes_path"]
Konstytucja_results_path = config["Konstytucja_results_path"][which_dataset]
xrf_path = config["xrf_path"]

## Loading the data & preprocessing

In [4]:
if dataset == 'Konstytucja_indicators':
    input_data = np.loadtxt(target_path, delimiter=',', skiprows=1, usecols=range(19))
    colnames = pd.read_csv(target_path, nrows=1, header=None)
    df = pd.DataFrame(data=input_data, columns=colnames.iloc[0,:-1])
    df = df[elements_to_keep]
elif dataset == 'Konstytucja_prediction':
    input_data = np.loadtxt(Konstytucja_results_path, delimiter=',')
    df = pd.DataFrame(data=input_data, columns=elements_to_keep)
elif dataset == 'XRF':
    input_data = np.loadtxt(xrf_path, delimiter=',', skiprows=1, usecols=(2,3,4))
    df = pd.DataFrame(data=input_data, columns=elements_to_keep_xrf)
    ground_truth_df = pd.read_csv(xrf_path)[['name', 'short', 'OPIS']]
    ground_truth_df.rename(columns={'name': 'Sample_id'}, inplace=True)
    ground_truth_df.reset_index(drop=True, inplace=True)

df.reset_index(drop=True, inplace=True)

In [5]:
classes_df = pd.read_excel(classes_path, header=1, usecols=['NAZWA', 'OPIS'])
classes_df['NAZWA_short'] = classes_df['NAZWA'].apply(lambda x: 
                                                    re.split(r'(\d+)', x)[0] + re.split(r'(\d+)', x)[1] if len(re.split(r'(\d+)', x))>1 else re.split(r'(\d+)', x)[0]
                                                   )                                                    
classes_df.drop_duplicates(['NAZWA', 'OPIS'], inplace=True)
classes_df_supp = classes_df.drop_duplicates(['NAZWA_short'])
classes_df_supp['NAZWA'] = classes_df['NAZWA_short']
classes_df = pd.concat([classes_df, classes_df_supp])
classes_df.drop_duplicates(inplace=True)
classes_df.reset_index(inplace=True, drop=True)

/tmp/ipykernel_28895/3014040627.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  classes_df_supp['NAZWA'] = classes_df['NAZWA_short']


In [6]:
def remove_outer(group, n):
    return group.iloc[n:-n] if len(group) > 2*n else pd.DataFrame(columns=group.columns)

In [7]:
########################################################
# check if joining is ok: especially labelss like APP15D and similar (if they have nans and if they should have nans)

In [8]:
if dataset == 'Konstytucja_indicators' or dataset == 'Konstytucja_prediction':
    
    if which_dataset == 'old':
        ground_truth_df = pd.read_csv(target_path, usecols=['probka', 'short'])
        ground_truth_df.rename(columns={"probka": "Sample_id"}, inplace=True)
    elif which_dataset == 'new':
        ground_truth_df = pd.read_csv(target_path, usecols=['name'])
        ground_truth_df['Sample_id'] = ground_truth_df['name'].apply(lambda x: x.split('_')[0] if len(x.split('_'))==2 else x.split('_')[0] + x.split('_')[1])
        ground_truth_df['Sample_id'] = ground_truth_df['Sample_id'].apply(lambda x: x.replace('.', ''))
        ground_truth_df['short'] = ground_truth_df['Sample_id'].apply(lambda x: re.split(r'\d+', x)[0])
        ground_truth_df.drop(columns=['name'], inplace=True)
    
    ground_truth_df.reset_index(drop=True, inplace=True)
    
    if remove_outer_bool:
        ground_truth_df = ground_truth_df.groupby('Sample_id', group_keys=False).apply(remove_outer, n=how_many_outer_to_remove)

In [9]:
if dataset == 'Konstytucja_indicators' or dataset == 'Konstytucja_prediction':

    ground_truth_df = pd.merge(left=ground_truth_df, right=classes_df, how='left', left_on='Sample_id', right_on='NAZWA')
    
    ground_truth_df['OPIS'] = ground_truth_df['OPIS'].apply(lambda x: str(x).strip())
    ground_truth_df.drop(columns=['NAZWA', 'NAZWA_short'], inplace=True)
    ground_truth_df.reset_index(drop=True, inplace=True)

In [10]:
# with pd.option_context('display.max_rows', None, 'display.max_columns', None):
#     print(ground_truth_df)

### Dividing to APP, ASC, ML

In [11]:
df_app = df[ground_truth_df['short'] == 'APP']
df_asc = df[ground_truth_df['short'] == 'ASC']
df_ml = df[ground_truth_df['short'] == 'ML']

if book_name == 'app':
    df = df_app
elif book_name == 'asc':
    df = df_asc
elif book_name == 'ml':
    df = df_ml
elif book_name == 'all':
    pass

### Removing some data

1. Let's keep only columns that we need.

To reduce the set of used elements run cell below. Then, instead of predicting 29 numbers, we will predict only 8. We will also use only 8 numbers as input.

In [12]:
if dataset == 'Konstytucja_indicators' or dataset == 'Konstytucja_prediction':
    df = df[elements_to_keep]

2. Let's check if there are any rows with missing values.

In [13]:
(df.shape[0] - df.dropna().shape[0])/df.shape[0]

0.0

### Converting to np.array

In [14]:
X = np.array(df.values)

### Normalizing / taking logarithm

In [15]:
def adjusted_log_transform(input_array):
    res = np.where(input_array>0, np.log(input_array), 0.)
    return res

In [16]:
if dataset == 'Konstytucja_indicators' or dataset == 'Konstytucja_prediction':

    if preprocessing_method == 'normalization':
    
        X = (X - np.min(X, axis=0))/np.std(X, axis=0)
        
    elif preprocessing_method == 'logarithm':
        
        X = adjusted_log_transform(X)
    
    elif preprocessing_method == 'logarithm_and_normalization':
        
        #logarithm
        X = adjusted_log_transform(X)
        
        #normalization
        X = (X - np.min(X, axis=0))/np.std(X, axis=0)
        
    elif preprocessing_method == 'none':
        
        pass

/tmp/ipykernel_28895/2387706189.py:2: RuntimeWarning: invalid value encountered in log
  res = np.where(input_array>0, np.log(input_array), 0.)


### Ground truth

In [17]:
class_df_app = ground_truth_df[ground_truth_df['short'] == 'APP'][['Sample_id','OPIS']]
class_df_asc = ground_truth_df[ground_truth_df['short'] == 'ASC'][['Sample_id', 'OPIS']]
class_df_ml = ground_truth_df[ground_truth_df['short'] == 'ML'][['Sample_id', 'OPIS']]

if book_name == 'app':
    class_df = class_df_app
elif book_name == 'asc':
    class_df = class_df_asc
elif book_name == 'ml':
    class_df = class_df_ml
elif book_name == 'all':
    class_df = ground_truth_df[['Sample_id', 'OPIS']]

class_df.columns = ['Code', 'Class name']

In [18]:
y = class_df['Class name']
y = [str(y) for y in y]
class_df.reset_index(inplace=True, drop=True)

In [19]:
class_df_signatures = class_df[class_df['Class name'].apply(lambda x: 'podpis' in x)]

In [20]:
X_signatures = X[class_df['Class name'].apply(lambda x: 'podpis' in x), :]

### Quality of clusters

In [21]:
silhouette_score(X=X_signatures, labels=class_df_signatures["Code"])

np.float64(-0.005894062194422203)

In [22]:
# XRF Code

# app -0.14
# asc 0.39
# ml -0.20

# Konstytucja indicators Code

# app 0.20
# asc 0.44
# ml 0.13

# Konstytucja prediction Code

# app -0.01
# asc 0.19
# ml -0.07

In [23]:
########################################################

In [24]:
codes_to_nrs_dict = {key: i for i, key in enumerate(class_df_signatures["Code"].unique())}

In [25]:
nr_labels = class_df_signatures["Code"].replace(codes_to_nrs_dict)

/tmp/ipykernel_28895/985231896.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  nr_labels = class_df_signatures["Code"].replace(codes_to_nrs_dict)
